In [1]:
import os
os.chdir("..")

In [2]:
import numpy as np 
from time import perf_counter
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from structures.layer import Layer
from structures.neuron import Neuron
from structures.activation_functions import Sigmoid
from structures.metrics import evaluate_accuracy

## Implementation of simple feedforward neural network

In [3]:
# Generate dataset
X, y = make_classification(n_samples=500, n_features=5, n_informative=5,
                           n_redundant=0, n_clusters_per_class=2, random_state=42)
y = y.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Hyperparameters
epochs = 200
lr = 0.1

### Standard OOP approach - Neuron and Layer classes

In [5]:
# Create network
input_size = X.shape[1]
l0 = Layer([Neuron(input_shape=input_size, activation_function=Sigmoid()) for _ in range(10)])
l1 = Layer([Neuron(input_shape=10, activation_function=Sigmoid()) for _ in range(20)])
l2 = Layer([Neuron(input_shape=20, activation_function=Sigmoid())])
network = [l0, l1, l2]

In [6]:
# Train network
t1 = perf_counter()

for epoch in range(epochs):
    loss = 0
    for xi, yi in zip(X_train, y_train):
        # Forward
        out0 = l0.activate(xi)
        out1 = l1.activate(out0)
        out2 = l2.activate(out1)

        # Loss
        error = out2[0] - yi
        loss += 0.5 * error ** 2

        # Backward
        delta2 = np.array(error)
        delta1 = l1.compute_deltas(l2.get_weights_matrix(), delta2)
        delta0 = l0.compute_deltas(l1.get_weights_matrix(), delta1)

        # Update
        l2.backward(delta2, lr)
        l1.backward(delta1, lr)
        l0.backward(delta0, lr)

    if epoch % 20 == 0 or epoch == epochs - 1:
        acc = evaluate_accuracy(network, X_test, y_test)
        print(f"Epoch {epoch} Loss: {loss.item():.6f}, Accuracy: {acc:.2f}")
        
t2 = perf_counter()
print(f"Exec. time: {round(t2 - t1, 4)}s.")

Epoch 0 Loss: 53.257036, Accuracy: 0.53
Epoch 20 Loss: 14.065690, Accuracy: 0.83
Epoch 40 Loss: 8.852493, Accuracy: 0.85
Epoch 60 Loss: 7.448769, Accuracy: 0.86
Epoch 80 Loss: 6.431313, Accuracy: 0.85
Epoch 100 Loss: 4.935552, Accuracy: 0.85
Epoch 120 Loss: 4.090126, Accuracy: 0.86
Epoch 140 Loss: 3.491176, Accuracy: 0.86
Epoch 160 Loss: 3.028149, Accuracy: 0.85
Epoch 180 Loss: 2.790126, Accuracy: 0.83
Epoch 199 Loss: 2.663524, Accuracy: 0.82
Exec. time: 18.3765s.


---

### Optimized Layer logic
- Drop Neuron class
- Layer holds a weight matrix and activation function directly
- Forward and backward passes become matrix operations

In [7]:
from structures.optimized_layer import OptimizedLayer

In [8]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    fx = sigmoid(x)
    return fx * (1 - fx)

In [9]:
input_size = X.shape[1]
l0 = OptimizedLayer(input_size, 10, sigmoid, sigmoid_deriv)
l1 = OptimizedLayer(10, 20, sigmoid, sigmoid_deriv)
l2 = OptimizedLayer(20, 1, sigmoid, sigmoid_deriv)
network = [l0, l1, l2]

In [10]:
t1 = perf_counter()

for epoch in range(epochs):
    loss = 0
    for xi, yi in zip(X_train, y_train):
        # Forward
        out0 = l0.activate(xi)
        out1 = l1.activate(out0)
        out2 = l2.activate(out1)

        # Loss
        error = out2[0] - yi
        loss += 0.5 * error ** 2

        # Backward
        delta2 = np.array(error)
        delta1 = l1.compute_deltas(l2.weights, delta2)
        delta0 = l0.compute_deltas(l1.weights, delta1)

        # Update
        l2.backward(delta2, lr)
        l1.backward(delta1, lr)
        l0.backward(delta0, lr)

    if epoch % 20 == 0 or epoch == epochs - 1:
        acc = evaluate_accuracy(network, X_test, y_test)
        print(f"Epoch {epoch} Loss: {loss.item():.6f}, Accuracy: {acc:.2f}")
        
t2 = perf_counter()
print(f"Exec. time: {round(t2 - t1, 4)}s.")

Epoch 0 Loss: 53.276938, Accuracy: 0.53
Epoch 20 Loss: 13.446616, Accuracy: 0.83
Epoch 40 Loss: 8.851392, Accuracy: 0.86
Epoch 60 Loss: 6.923311, Accuracy: 0.86
Epoch 80 Loss: 5.652825, Accuracy: 0.88
Epoch 100 Loss: 4.732954, Accuracy: 0.87
Epoch 120 Loss: 4.159901, Accuracy: 0.87
Epoch 140 Loss: 3.596125, Accuracy: 0.86
Epoch 160 Loss: 3.222795, Accuracy: 0.86
Epoch 180 Loss: 2.963644, Accuracy: 0.84
Epoch 199 Loss: 2.809451, Accuracy: 0.84
Exec. time: 3.1802s.


---

### Vectorized approach

In [11]:
from structures.network import VectorizedNetwork

In [12]:
layer_sizes = [5, 10, 20, 1]
vector_net = VectorizedNetwork(layer_sizes, sigmoid, sigmoid_deriv)

In [13]:
t1 = perf_counter()

for epoch in range(epochs):
    loss = 0
    for xi, yi in zip(X_train, y_train):
        vector_net.backward(xi, yi, lr)
        pred = vector_net.forward(xi)[0][-1][0]
        loss += 0.5 * (pred - yi[0]) ** 2

    if epoch % 20 == 0 or epoch == epochs - 1:
        acc = vector_net.evaluate(X_test, y_test)
        print(f"Epoch {epoch} Loss: {loss.item():.6f}, Accuracy: {acc:.2f}")
        
t2 = perf_counter()
print(f"Exec. time: {round(t2 - t1, 4)}s.")

Epoch 0 Loss: 47.427586, Accuracy: 0.53
Epoch 20 Loss: 48.170697, Accuracy: 0.53
Epoch 40 Loss: 48.462117, Accuracy: 0.53
Epoch 60 Loss: 20.941740, Accuracy: 0.78
Epoch 80 Loss: 13.543421, Accuracy: 0.83
Epoch 100 Loss: 11.515136, Accuracy: 0.83
Epoch 120 Loss: 10.923872, Accuracy: 0.83
Epoch 140 Loss: 10.568786, Accuracy: 0.83
Epoch 160 Loss: 10.314272, Accuracy: 0.83
Epoch 180 Loss: 10.050942, Accuracy: 0.84
Epoch 199 Loss: 9.448577, Accuracy: 0.86
Exec. time: 4.661s.
